# Baseline — CIFAR-10 Image Colorization (from scratch)

**Competition:** given a **grayscale** 32×32 CIFAR-10 image, predict the original RGB
image. **No pretrained models allowed** — everything is trained from scratch.

- **Task:** predict R,G,B values (0–255) for every pixel of 2000 test images
- **Submission:** one row per pixel/channel: `imageid_y_x_channel,value`
- **Metric:** pixel-level error (lower is better)
- **Kaggle link:** _TODO: add link_

**Approach:** a small convolutional encoder–decoder trained from scratch to map the
gray channel to RGB. A trivial but important reference: predicting the gray value for
all three channels already gets the *brightness* right — our CNN must beat that.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image

DATA_DIR = "."
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")
print(train_df.shape, test_df.shape)

def load(paths):
    return np.stack([np.asarray(Image.open(f"{DATA_DIR}/{p}")) for p in paths])

Xg = load(train_df["gray_path"]).astype(np.float32) / 255.0          # (N,32,32)
Yc = load(train_df["color_path"]).astype(np.float32) / 255.0         # (N,32,32,3)
Tg = load(test_df["gray_path"]).astype(np.float32) / 255.0
print(Xg.shape, Yc.shape, Tg.shape)

(20000, 5) (2000, 4)


(20000, 32, 32) (20000, 32, 32, 3) (2000, 32, 32)


In [2]:
# Baseline reference: gray replicated to RGB
val_idx = np.arange(len(Xg))[-2000:]         # last 2000 images = validation
tr_idx  = np.arange(len(Xg))[:-2000]
gray_mae = np.abs(np.repeat(Xg[val_idx][..., None], 3, -1) - Yc[val_idx]).mean() * 255
print(f"Gray-replication MAE: {gray_mae:.2f} (out of 255)")

Gray-replication MAE: 12.87 (out of 255)


In [3]:
class ColorNet(nn.Module):
    # Predicts a color *correction* on top of the gray image (skip connection):
    # output = gray + delta. Starting from the gray-replication baseline means
    # the network only has to learn the color shift, not the whole image.
    def __init__(self, ch=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, ch, 3, padding=1), nn.ReLU(),
            nn.Conv2d(ch, ch, 3, padding=1), nn.ReLU(),
            nn.Conv2d(ch, ch, 3, padding=1), nn.ReLU(),
            nn.Conv2d(ch, ch, 3, padding=1), nn.ReLU(),
            nn.Conv2d(ch, 3, 3, padding=1), nn.Tanh(),
        )
    def forward(self, x):  # x: (B,1,32,32) -> (B,3,32,32)
        return (x.repeat(1, 3, 1, 1) + 0.5 * self.net(x)).clamp(0, 1)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = ColorNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_fn = nn.L1Loss()

Xtr = torch.tensor(Xg[tr_idx]).unsqueeze(1)
Ytr = torch.tensor(Yc[tr_idx]).permute(0, 3, 1, 2)
Xva = torch.tensor(Xg[val_idx]).unsqueeze(1)
Yva = torch.tensor(Yc[val_idx]).permute(0, 3, 1, 2)

EPOCHS, BS = 6, 256
for ep in range(EPOCHS):
    model.train(); perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), BS):
        idx = perm[i:i+BS]
        opt.zero_grad()
        loss = loss_fn(model(Xtr[idx].to(device)), Ytr[idx].to(device))
        loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        va_mae = sum(loss_fn(model(Xva[i:i+BS].to(device)), Yva[i:i+BS].to(device)).item()
                     * len(Xva[i:i+BS]) for i in range(0, len(Xva), BS)) / len(Xva) * 255
    print(f"epoch {ep+1}: val MAE {va_mae:.2f} / 255")

epoch 1: val MAE 12.34 / 255


epoch 2: val MAE 12.29 / 255


epoch 3: val MAE 12.27 / 255


epoch 4: val MAE 12.25 / 255


epoch 5: val MAE 12.26 / 255


epoch 6: val MAE 12.20 / 255


In [4]:
# Predict test images and build the per-pixel submission
model.eval()
with torch.no_grad():
    out = torch.cat([model(torch.tensor(Tg[i:i+256]).unsqueeze(1).to(device)).cpu()
                     for i in range(0, len(Tg), 256)])
pred = (out.permute(0, 2, 3, 1).numpy() * 255).round().clip(0, 255).astype(np.uint8)

ids = test_df["id"].values
H = W = 32
rows = np.empty(len(ids) * H * W * 3, dtype=object)
vals = np.empty(len(ids) * H * W * 3, dtype=np.int64)
k = 0
for n, img_id in enumerate(ids):
    for yy in range(H):
        for xx in range(W):
            for ci, cname in enumerate("RGB"):
                rows[k] = f"{img_id}_{yy}_{xx}_{cname}"
                vals[k] = pred[n, yy, xx, ci]
                k += 1
sub = pd.DataFrame({"row_id": rows, "value": vals})
sub.to_csv("submission.csv", index=False)
print(sub.shape); sub.head()

(6144000, 2)


,row_id,value
0,test_00000_0_0_R,119
1,test_00000_0_0_G,120
2,test_00000_0_0_B,116
3,test_00000_0_1_R,65
4,test_00000_0_1_G,67


## Ideas to improve

- Train longer with a **U-Net** (skip connections) and more channels — on a GPU.
- Predict in **Lab color space**: keep the L channel from the input, predict only a/b.
- Colorization is multi-modal (a car can be red or blue): classification over color
  bins with class rebalancing (Zhang et al. 2016) beats plain regression.
- Add a perceptual or adversarial loss for more vivid colors.
